### Imports and Hyperparameter Settings

In [ ]:
%load_ext autoreload
%autoreload 2

from multimodal_mazes.evolution.genomes.genome import Genome
from multimodal_mazes.evolution.algorithms.genome_EA import GenomeEA

import numpy as np
import unittest
import copy

In [ ]:
HYPERPARAMETERS = {
    'task': 'maze',
    'n_module_types': 3,
    'n_inputs': 8,
    'n_outputs': 4,
    'n_modules': 4,
    'weight_sharing': False,
    'uniform_weights': False, # For weight sharing
    'connectivity': 'UNCONNECTED', # Options: 'FULLY CONNECTED', 'UNCONNECTED', 'SPARSE', 'RANDOM'
    'connection_density' : {'input_density': None, 'output_density': None}, # For 'RANDOM' connectivity
    'population_size': 80,
    'top_genomes': 20, 
    'mutation_rate': 0.8, 
    'crossover_rate': 0.5,
    'one_to_one': True # For UNCONNECTED, SPARSE, or RANDOM if density < 1.0
}

## Genome

### Unit Testing

In [ ]:
class TestGenome(unittest.TestCase):
    def setUp(self):
        """Set up the test case."""
        self.genome = Genome(genome_id=0, hyperparameters=HYPERPARAMETERS)

    def test_initialization(self):
        """
        Test the initialization of the genome.
        Tests:
            Genome ID is set correctly.
            Module groups and rules are initialized.
            Connection rules are initialized.
        """
        # Check genome properties
        self.assertEqual(self.genome.genome_id, 0)

        # Check module initialisation
        self.assertEqual(len(self.genome.mod_grouped_in), HYPERPARAMETERS['n_modules'])
        self.assertEqual(len(self.genome.modules), HYPERPARAMETERS['n_modules'])
        self.assertEqual(len(self.genome.mod_grouped_out), HYPERPARAMETERS['n_outputs'])
        
        # Check connection rules
        self.assertEqual(self.genome.conn_in_rules, [])
        self.assertEqual(self.genome.conn_out_rules, [])
        
    def test_forward_pass(self):
        """
        Test the forward pass of the genome.
        Tests:
            Output is a numpy array.
            Output shape is correct.
        """
        input_vec = np.ones(self.genome.n_inputs)
        output_vec = self.genome.forward_pass(input_vec)

        # Check output
        self.assertIsInstance(output_vec, np.ndarray)
        self.assertEqual(output_vec.shape[0], self.genome.n_outputs)

    def test_crossover(self):
        """
        Test the crossover of the genome.
        Tests:
            Child genome is created.
            Child genome has correct ID.
            Child genome has compile flag set.
            Child genome inherits properties from parents.
        """
        parent_1 = Genome(genome_id=1,hyperparameters=HYPERPARAMETERS)
        parent_2 = Genome(genome_id=2, hyperparameters=HYPERPARAMETERS)
        child = parent_1.crossover(new_id=3, parent_2=parent_2)

        # Check child genome properties
        self.assertIsInstance(child, Genome)
        self.assertEqual(child.genome_id, 3)
        self.assertEqual(child.compile_flag, 1)

        # Check inheritance from parents
        if len(parent_1.conn_in_rules) > 0 and len(parent_2.conn_in_rules) > 0:
            self.assertNotEqual(child.conn_in_rules, parent_1.conn_in_rules)
            self.assertNotEqual(child.conn_out_rules, parent_2.conn_out_rules)

    def test_mutation(self):
        """
        Test the mutation of the genome.
        Tests:
            Mutation does not alter modules
            Mutation alters the genome's connection rules.
        """
        mut_genome = Genome(genome_id=5, hyperparameters=HYPERPARAMETERS)
        org_in_rules = copy.deepcopy(mut_genome.conn_in_rules)
        org_out_rules = copy.deepcopy(mut_genome.conn_out_rules)
        for _ in range(10):
            mut_genome.mutate()

        # Check mutation effects
        self.assertNotEqual(mut_genome.conn_in_rules, org_in_rules)
        self.assertNotEqual(mut_genome.conn_out_rules, org_out_rules)

    def test_clone(self):
        """
        Test the cloning of the genome.
        Tests:
            Cloned genome is created.
            Cloned genome has correct ID.
            Cloned genome has compile flag set.
            Cloned genome inherits properties from original.
        """
        cloned_genome = self.genome.clone(new_id=4)

        # Check cloned genome properties
        self.assertIsInstance(cloned_genome, Genome)
        self.assertEqual(cloned_genome.genome_id, 4)
        self.assertEqual(self.genome.genome_id, 0)
        self.assertEqual(cloned_genome.compile_flag, 1)

        # Check inheritance from original
        for rule1, rule2 in zip(cloned_genome.conn_in_rules, self.genome.conn_in_rules):
            self.assertTupleEqual(rule1, rule2)
        for rule1, rule2 in zip(cloned_genome.conn_out_rules, self.genome.conn_out_rules):
            self.assertTupleEqual(rule1, rule2)

In [ ]:
genome_suite = unittest.TestLoader().loadTestsFromTestCase(TestGenome)
unittest.TextTestRunner(verbosity=2).run(genome_suite)

### Output Inspection

#### Initialisation

In [ ]:
genome = Genome(genome_id=0, hyperparameters=HYPERPARAMETERS)

print("Genome ID:", genome.genome_id)

print("\nInput Connect Rules:")
for rule in genome.conn_in_rules:
    print(rule)

print("\nOutput Connect Rules:")
for rule in genome.conn_out_rules:
    print(rule)

genome.plot_genome()

#### Connectivity

In [ ]:
genome_c = Genome(genome_id=0, hyperparameters=HYPERPARAMETERS)
genome_c.plot_genome()

#### Crossover

In [ ]:
parent_1 = Genome(genome_id=1, hyperparameters=HYPERPARAMETERS)
parent_2 = Genome(genome_id=2, hyperparameters=HYPERPARAMETERS)
child = parent_1.crossover(new_id=3, parent_2=parent_2)

print("Parent 1 Input Connection Rules:"  )
for rule in parent_1.conn_in_rules:
    print(rule)

print("\nParent 1 Output Connection Rules:")
for rule in parent_1.conn_out_rules:
    print(rule)

print("\nParent 2 Input Connection Rules:")
for rule in parent_2.conn_in_rules:
    print(rule)

print("\nParent 2 Output Connection Rules:")
for rule in parent_2.conn_out_rules:
    print(rule)

print("\nChild Input Connection Rules:")
for rule in child.conn_in_rules:
    print(rule)

print("\nChild Output Connection Rules:")
for rule in child.conn_out_rules:
    print(rule)

In [ ]:
parent_1.plot_genome()
parent_2.plot_genome()
child.plot_genome()

#### Mutation

In [ ]:
genome_mut = Genome(genome_id=0, hyperparameters=HYPERPARAMETERS)

print("Original Input Connection Rules:")
for rule in genome_mut.conn_in_rules:
    print(rule)

print("\nOriginal Output Connection Rules:")
for rule in genome_mut.conn_out_rules:
    print(rule)

# genome_mut.plot_genome()
genome_mut.mutate()
# genome_mut.plot_genome()

print("\nMutated Input Connection Rules:")
for rule in genome_mut.conn_in_rules:
    print(rule)

print("\nMutated Output Connection Rules:")
for rule in genome_mut.conn_out_rules:
    print(rule)

#### Forward Pass

In [ ]:
genome_fp = Genome(genome_id=0, hyperparameters=HYPERPARAMETERS)
genome_fp.plot_genome()

In [ ]:
input_vec = np.array([0.0, 0.0, 1.0, 0.0, 0.5, 0.0, 0.0, 0.0])
print("Input Vector:", input_vec)

output_vec = genome_fp.forward_pass(input_vec)
print("Output Vector:", output_vec)

input_vec = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
print("Input Vector:", input_vec)

output_vec = genome_fp.forward_pass(input_vec)
print("Output Vector:", output_vec)

#### Compile

In [ ]:
genome_c = Genome(genome_id=0, hyperparameters=HYPERPARAMETERS)
genome_c.plot_genome()

In [ ]:
genome_c.compile_flag = 1
genome_c.compile_rules()

print("Input connectivity rules:")
print(genome_c.mod_grouped_in)

print("Output connectivity rules:")
print(genome_c.mod_grouped_out)

## GenomeEA

### Unit Testing

In [ ]:
class TestGenomeEA(unittest.TestCase):
    def setUp(self):
        """Set up the test case."""
        self.ea = GenomeEA(hyperparameters=HYPERPARAMETERS)

    def test_initialization(self):
        """
        Test the initialization of the GenomeEA class.
        Tests:
            The generation counter is initialized.
            The genomes list is initialized.
            The population size is set correctly.
            The last genome ID is initialized.
            The fittest networks list is initialized.
            The fittest network is initialized.
        """
        # Check generation initialization
        self.assertEqual(self.ea.generation, 0)

        # Check genome initialization
        self.assertIsInstance(self.ea.genomes, list)
        self.assertEqual(len(self.ea.genomes), 0)
        self.assertEqual(self.ea.population, HYPERPARAMETERS['population_size'])
        self.assertEqual(self.ea.last_genome_id, 0)
        self.assertIsInstance(self.ea.fittest_networks, list)
        self.assertEqual(len(self.ea.fittest_networks), 0)
        self.assertEqual(self.ea.fittest_network, None)
        
    def test_generate(self):
        """
        Test the generate method of the GenomeEA class.
        Tests:
            The genomes list is populated.
            The fittest network is updated.
            The fittest networks list is updated.
            The first genome is of the correct type.
        """
        self.ea.generate()

        # Check genome population properties
        self.assertNotEqual(len(self.ea.genomes), 0)
        self.assertEqual(self.ea.fittest_network, self.ea.genomes[0])
        self.assertEqual(self.ea.fittest_networks, self.ea.genomes[:HYPERPARAMETERS['top_genomes']])
        self.assertIsInstance(self.ea.genomes[0], Genome)
        
    def test_evolve(self):
        """
        Test the evolve method of the GenomeEA class.
        Tests:
            The generation counter is incremented.
            The genomes list is updated.
            The last genome ID is updated.
        """
        self.ea.generate()
        original_generation = copy.deepcopy(self.ea.generation)
        original_genomes = copy.deepcopy(self.ea.genomes)
        original_last_genome_id = copy.deepcopy(self.ea.last_genome_id)
        self.ea.evolve()

        # Check evolution properties
        self.assertEqual(self.ea.generation, original_generation + 1)
        self.assertEqual(len(self.ea.genomes), HYPERPARAMETERS['population_size'])

        # Check genome updates
        self.assertNotEqual(self.ea.genomes, original_genomes)
        self.assertGreaterEqual(self.ea.last_genome_id, original_last_genome_id + HYPERPARAMETERS['population_size'] - HYPERPARAMETERS['top_genomes'])

In [ ]:
ea_suite = unittest.TestLoader().loadTestsFromTestCase(TestGenomeEA)
unittest.TextTestRunner(verbosity=2).run(ea_suite)